# 02 — Baseline characteristics and pre-intervention balance

Reproduces Table 4 (sample characteristics) and Tables S2-S3 (pre-intervention
gambling outcomes by arm), reported descriptively per current CONSORT guidance
(no significance testing of baseline/randomization balance).

In [ ]:
import pandas as pd
import numpy as np

ARM_COL = "CONTROL_GROUP"
ARM_MAP = {"AA": "A", "BB": "B", "CC": "C", "EE": "E", "FF": "F", "CONTROLGROUP": "Control"}

OUTCOMES = {
    "SUM_DEPOSITS_W": "Sum deposits", "NR_DEPOSITS_W": "Number deposits",
    "SUM_WAGERS_DAILY_W": "Sum wagers", "N_WAGERS_DAILY_W": "Number wagers",
    "NR_DAYS_W": "Gambling days", "SUM_TL_W": "Theoretical loss",
}

def pre_sum(df, prefix):
    cols = [f"{prefix}_PRE_{w}" for w in range(1, 13)]
    cols = [c for c in cols if c in df.columns]
    return df[cols].sum(axis=1, min_count=1)

In [ ]:
def sample_characteristics(path, label, arms):
    df = pd.read_csv(path, low_memory=False)
    df["Arm"] = df[ARM_COL].map(ARM_MAP).fillna(df[ARM_COL])

    print("#" * 70); print(f"#  {label}  (N={len(df)})"); print("#" * 70)

    if "BIRTH_YEAR" in df.columns:
        df["AGE"] = 2022 - df["BIRTH_YEAR"]
        print("\nMedian age:", df["AGE"].median())

    sexcol = next((c for c in ["GENDER", "SEX"] if c in df.columns), None)
    if sexcol:
        male_pct = (df[sexcol] == 1).mean() * 100  # confirm coding against data dictionary
        print(f"Male (%): {male_pct:.1f}")

    print("\n--- Pre-intervention gambling outcomes (median by arm) ---")
    rows = []
    for prefix, name in OUTCOMES.items():
        y = pre_sum(df, prefix)
        sub = pd.DataFrame({"arm": df["Arm"], "y": y}).dropna()
        meds = sub.groupby("arm")["y"].median().reindex(arms + ["Control"]).round(1)
        rows.append({"Outcome": name, **meds.to_dict()})
    print(pd.DataFrame(rows).to_string(index=False))

sample_characteristics("P10_final.csv", "Experiment 1", ["A", "C", "E", "F"])
print()
sample_characteristics("P11_final.csv", "Experiment 2", ["A", "B", "C"])